<a href="https://colab.research.google.com/github/AhmedEssam1996/AhmedEssam1996/blob/main/cv_ocr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade pillow pdf2image transformers gradio accelerate
!apt-get install -y poppler-utils

In [ ]:
import os
import torch
import gc
import traceback
from PIL import Image
from pdf2image import convert_from_path
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import gradio as gr

# ===============================
# 1. Engine Initialization
# ===============================
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

def clean_memory():
    """Force memory cleanup to prevent Out-Of-Memory errors"""
    gc.collect()
    torch.cuda.empty_cache()

# ===============================
# 2. Forensic Core
# ===============================

def run_analysis(file, progress=gr.Progress()):
    if file is None:
        return ["Error: No file detected."] * 4

    clean_memory()

    if isinstance(file, str):
        path_str = file
    elif hasattr(file, 'name'):
        path_str = str(file.name)
    elif isinstance(file, dict) and 'path' in file:
        path_str = file['path']
    else:
        path_str = str(file)

    try:
        progress(0.2, desc="Converting Document...")
        if path_str.lower().endswith(".pdf"):
            pages = convert_from_path(path_str, dpi=100)
            target_img = pages[0].convert("RGB")
        else:
            target_img = Image.open(path_str).convert("RGB")

        # --- Phase 1: Visual Audit & Data Extraction ---
        progress(0.4, desc="Visual & OCR Engine Running...")

        vision_query = (
            "1. Extract all text accurately.\n"
            "2. Rate the professional design layout out of 10.\n"
            "3. Identify the job title this person is seeking."
        )

        msgs_vision = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": vision_query}
                ]
            }
        ]

        v_prompt = processor.apply_chat_template(msgs_vision, tokenize=False, add_generation_prompt=True)
        v_inputs = processor(text=[v_prompt], images=[target_img], padding=True, return_tensors="pt").to(model.device)

        with torch.no_grad():
            v_gen = model.generate(**v_inputs, max_new_tokens=800)

        raw_text = processor.batch_decode(v_gen[:, v_inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
        visual_audit = f"### Design Review & Extraction\n\n{raw_text}"

        clean_memory()

        # --- Phase 2: Hiring Simulator (Isolated Call) ---
        progress(0.7, desc="Running Performance Simulation...")

        sim_query = f"Resume Data:\n{raw_text[:1500]}\n\nTask: Provide a 'Hiring Simulator' report. What is the biggest risk and biggest potential gain of hiring this candidate? Be highly analytical and concise."
        msgs_sim = [{"role": "user", "content": sim_query}]

        s_prompt = processor.apply_chat_template(msgs_sim, tokenize=False, add_generation_prompt=True)
        s_inputs = processor(text=[s_prompt], return_tensors="pt").to(model.device)

        with torch.no_grad():
            s_gen = model.generate(**s_inputs, max_new_tokens=300)

        sim_report = processor.batch_decode(s_gen[:, s_inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

        clean_memory()

        # --- Phase 3: ATS Strategy (Isolated Call) ---
        progress(0.9, desc="Calculating ATS Optimization...")

        ats_query = f"Resume Data:\n{raw_text[:1500]}\n\nTask: Provide an 'ATS Strategy' report. List 5 missing technical keywords or formatting issues that would prevent this resume from passing an automated Applicant Tracking System. Keep it formatted as a list."
        msgs_ats = [{"role": "user", "content": ats_query}]

        a_prompt = processor.apply_chat_template(msgs_ats, tokenize=False, add_generation_prompt=True)
        a_inputs = processor(text=[a_prompt], return_tensors="pt").to(model.device)

        with torch.no_grad():
            a_gen = model.generate(**a_inputs, max_new_tokens=300)

        ats_report = processor.batch_decode(a_gen[:, a_inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

        progress(1.0, desc="Analysis Complete")
        return visual_audit, sim_report, ats_report, raw_text

    except Exception as e:
        error_details = traceback.format_exc()
        error_msg = f"SYSTEM FAILURE:\n{str(e)}\n\nTraceback Details:\n{error_details}"
        return [error_msg] * 4

# ===============================
# 3. High-End Dashboard UI
# ===============================

with gr.Blocks(theme=gr.themes.Default(primary_hue="slate")) as demo:
    gr.Markdown("# AI RESUME WAR ROOM")
    gr.Markdown("Visual Intelligence & Performance Prediction Engine")

    with gr.Row():
        with gr.Column(scale=1):
            file_up = gr.File(label="Target Resume", type="filepath")
            btn = gr.Button("EXECUTE ANALYSIS", variant="primary")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("Visual Audit"):
                    out_v = gr.Markdown("Ready for processing...")
                with gr.TabItem("Hiring Simulator"):
                    out_s = gr.Markdown("Ready for processing...")
                with gr.TabItem("ATS Shadowing"):
                    out_a = gr.Markdown("Ready for processing...")
                with gr.TabItem("Source Text"):
                    out_r = gr.Textbox(label="Raw Extraction", lines=10)

    btn.click(run_analysis, inputs=file_up, outputs=[out_v, out_s, out_a, out_r])

if __name__ == "__main__":
    demo.launch(share=True, debug=True)